# Do Players Notice the AI, or Just the Quality?
**Author:** Neha Kollikara and Tomas Aguilar

**Course:** INFO 4340 - Analytics and AI

**Question:** Among buyers who dislike AI-disclosed Steam games, how often is generative AI the stated reason, compared with quality? (Q2, exploratory: when reviewers don't mention AI, do complaints concentrate where AI was used?)

**Notebook Parts:**
1. Pilot: keyword search and hand check (development set)
2. Clean: raw scrape to `data/reviews_clean.csv`
3. Keyword baseline *(coming)*
4. Claude classifier *(coming)*
5. Evaluation *(coming)*
6. Analysis *(coming)*

# Setup

In [1]:
#Imports
import pandas as pd
import numpy as np
import os

# Part 1: Pilot
**Purpose:** Before the full scrape, estimate how often negative reviews of AI-disclosed games mention generative AI, and test a keyword search.

**Data:** 312 English negative reviews (Jan 2024 – Dec 2025) from the 5 lowest-rated AI-disclosed games at the time, scraped Nov 4, 2026.

**Outputs:** `data/pilot.csv` (development set, excluded from all testing) and `data/pilot_hits_labeled.csv` (hand-checked keyword hits).

**Key results:** 8 of 15 keyword hits (53%) were about generative AI; estimated rate 2.6% (95% CI 1.3–5.0%).

## Pull the Pilot

In [2]:
# Pull the pilot (already run on Nov 4, 2026; left commented out so re-running the notebook doesn't change the data)

# %run steam_reviews_scraper.py --appids 2738870 3484300 2687970 2943670 3444080 --max-per-game 100 --out data/pilot.csv

## Load and Check

In [3]:
# Load the pilot and confirm it matches what was scraped
pilot = pd.read_csv("data/pilot.csv")
print(f"Loaded: {pilot.shape[0]} rows, {pilot.shape[1]} columns")
print(pilot["appid"].value_counts())

assert len(pilot) == 312, f"Expected 312 pilot reviews, found {len(pilot)}"
assert pilot["appid"].nunique() == 5, "Pilot should cover 5 games"
assert (pilot["voted_up"] == False).all(), "Pilot should contain only negative reviews"
assert pilot["recommendationid"].is_unique, "Review IDs should be unique"

Loaded: 312 rows, 18 columns
appid
3484300    100
2687970    100
2738870     48
2943670     41
3444080     23
Name: count, dtype: int64


## Keyword Search

In [4]:
# Keyword search for AI mentions
# This pattern is fixed here and reused unchanged as the keyword baseline, so it isn't tuned on test data
AI_PATTERN = r"\bai\b|a\.i\.|ai-generated|generative|genai|slop|midjourney|chatgpt|artificial intelligence"

hits = pilot[pilot["review"].str.contains(AI_PATTERN, case=False, regex=True)].copy()
print(f"{len(hits)} of {len(pilot)} reviews match the keyword pattern ({len(hits)/len(pilot):.1%})")
print(hits.groupby("appid").size())

15 of 312 reviews match the keyword pattern (4.8%)
appid
2687970     3
2943670     1
3484300    11
dtype: int64


## Export for Hand-Check

In [5]:
# Export keyword hits for hand labeling
# Only writes the file if it doesn't exist, so re-running never erases the hand labels
out_file = "data/pilot_hits_labeled.csv"
if not os.path.exists(out_file):
    hits["label"] = ""
    hits["note"] = ""
    hits[["recommendationid", "appid", "review", "label", "note"]].to_csv(out_file, index=False)
    print(f"Exported {len(hits)} hits to {out_file}")
else:
    print(f"{out_file} already exists; not overwriting hand labels")

data/pilot_hits_labeled.csv already exists; not overwriting hand labels


## Hand-Check Results

In [6]:
# Hand check results
# Codes: G = generative AI, E = enemy/NPC/bot behavior, N = keyword matched but no AI meaning, ? = unclear
labeled = pd.read_csv("data/pilot_hits_labeled.csv")

assert len(labeled) == len(hits), "Every keyword hit should be in the labeled file"
assert labeled["label"].notna().all(), "Some hits are still unlabeled"
assert set(labeled["label"]) <= {"G", "E", "N", "?"}, f"Unexpected labels: {set(labeled['label'])}"

print(labeled["label"].value_counts())

def wilson_ci(k, n, z=1.96):
    """95% Wilson confidence interval for a proportion (works well for small counts)."""
    p = k / n
    center = (p + z**2 / (2*n)) / (1 + z**2 / n)
    half = z * np.sqrt(p*(1-p)/n + z**2 / (4*n**2)) / (1 + z**2 / n)
    return center - half, center + half

n_g = (labeled["label"] == "G").sum()
precision = n_g / len(labeled)
rate = n_g / len(pilot)
low, high = wilson_ci(n_g, len(pilot))
print(f"\nKeyword precision: {n_g} of {len(labeled)} hits ({precision:.0%}) were generative AI")
print(f"Estimated generative-AI rate: {rate:.1%} (95% CI {low:.1%} to {high:.1%})")

label
G    8
E    6
N    1
Name: count, dtype: int64

Keyword precision: 8 of 15 hits (53%) were generative AI
Estimated generative-AI rate: 2.6% (95% CI 1.3% to 5.0%)


## Investigate Edge Cases

In [7]:
# The non-G hits and their notes become examples for the labeling guide
for _, row in labeled[labeled["label"] != "G"].iterrows():
    print(f"[{row['label']}] {row['note'] if pd.notna(row['note']) else ''}\n   {row['review'][:150]}...\n")

[E] 
   This is not a fun game to play in its current state, there is some minor fun to be had with building a house and gear. But the human enemies are all S...

[E] 
   Just a generic survival wanna be game with stupid AI that just runs straight at you
the aiming is over sensitive no matter what settings you use ...fo...

[E] 
   (Update: It's better, but still way too buggy to recommend.)
Guns feels great, but the AI is so broken you'll just get aim botted from bushes before e...

[E] pretty sure it's E but AI mapping? What is that
   Well... let's start with the good, if you have a pretty beefy PC the game is gorgeous(def disable motion blur otherwise its a mess), The voice acting ...

[E] 
   I really wanted to like Dreadzone. It has a gritty, post-apocalyptic vibe concept that could’ve been a hit—but right out of the gate, the pacing felt ...

[E] 
   Let me start off by stating I actually really like the idea of this game a lot. The idea environment is pretty, but the sun rays a

## Pilot Decisions
- Complaints about enemy, NPC, or bot behavior ("the AI just runs at you") count as quality complaints, not generative AI complaints.
- Reviews are labeled by the reviewer's stated reason, so "AI random gen skins" counts as generative AI whether or not AI was actually used.
- "Slop" counts as generative AI only when AI is named or clearly implied.
- Keyword precision was 53%, so a keyword search alone is not reliable; this motivates the Claude classifier.
- These 312 reviews are the development set: the Claude prompt is tuned only on them, and they are removed from all test data.
- Bakery Cafe Simulator was in the pilot but later dropped from the final game list (only 23 negative reviews in the window).

---

# Part 2: Clean
**Purpose:** Clean the raw Steam review scrape, remove the pilot (development) reviews, and draw a random sample of up to 100 negative reviews per game for classification.

**Input:** `data/reviews_raw.csv` (raw scrape, Nov 4, 2026), `data/reviews_raw_replacements.csv` (2 replacement games), `data/games_final.csv` (30 games), `data/pilot.csv` (development set)

**Output:** `data/reviews_clean.csv`

**Summary:** *(filled in after cleaning)*

## Load the Data

In [8]:
#Read in the data and confirm
df = pd.read_csv("data/reviews_raw.csv")
games = pd.read_csv("data/games_final.csv")
pilot = pd.read_csv("data/pilot.csv")

print(f"Loaded reviews: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Loaded games: {games.shape[0]} rows")
print(f"Loaded pilot (dev set): {pilot.shape[0]} rows")

Loaded reviews: 6612 rows, 29 columns
Loaded games: 30 rows
Loaded pilot (dev set): 312 rows


## Explore the Data

In [9]:
#Explore the data structure
print("\nINFO:")
df.info()

print("\nDESCRIBE:")
print(df.describe().to_string())

print("\nDUPLICATES:")
print(f"Duplicate rows: {df.duplicated().sum()}")
print(f"Duplicate review IDs: {df['recommendationid'].duplicated().sum()}")

print("\nMISSING VALUES:")
print(df.isnull().sum())


INFO:
<class 'pandas.DataFrame'>
RangeIndex: 6612 entries, 0 to 6611
Data columns (total 29 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   appid                          6612 non-null   int64  
 1   recommendationid               6612 non-null   int64  
 2   review                         6612 non-null   str    
 3   voted_up                       6612 non-null   bool   
 4   votes_up                       6612 non-null   int64  
 5   votes_funny                    6612 non-null   int64  
 6   weighted_vote_score            6612 non-null   float64
 7   timestamp_created              6612 non-null   int64  
 8   language                       6612 non-null   str    
 9   steam_purchase                 6612 non-null   bool   
 10  received_for_free              6612 non-null   bool   
 11  written_during_early_access    6612 non-null   bool   
 12  refunded                       6612 non-null   bool 

In [10]:
#Explore the data: content checks specific to this scrape

# Date range: should fall within Jan 1, 2024 – Dec 31, 2025
print("DATE RANGE:")
created = pd.to_datetime(df["timestamp_created"], unit="s")
print(f"Earliest: {created.min()}, Latest: {created.max()}")

# Scrape filters: should be all negative (voted_up False) and all English
print("\nSCRAPE FILTER CHECKS:")
print(df["voted_up"].value_counts(dropna=False))
print(df["language"].value_counts(dropna=False))

# Reviews per game (smallest first) and whether all 30 games returned reviews
print("\nREVIEWS PER GAME:")
per_game = df.groupby(["group", "name"]).size().sort_values()
print(per_game.to_string())
print(f"\nGames with reviews: {df['appid'].nunique()} of {len(games)}")
missing_games = set(games["appid"]) - set(df["appid"])
print(f"Games with no reviews: {missing_games if missing_games else 'none'}")

# Reviews by group (disclosed vs. control)
print("\nREVIEWS BY GROUP:")
print(df["group"].value_counts())

# Pilot reviews in the scrape (dev set; removed later so they aren't used for testing)
print("\nPILOT (DEV SET) OVERLAP:")
in_pilot = df["recommendationid"].isin(pilot["recommendationid"])
print(f"Reviews also in pilot.csv: {in_pilot.sum()}")
print(df[in_pilot].groupby("name").size().to_string())

# Review types that may need handling: empty, very short, checkbox templates, non-English text
print("\nREVIEW TEXT:")
text = df["review"].fillna("").str.strip()
words = text.str.split().str.len()
print(f"Empty reviews: {(text == '').sum()}")
print(f"Very short reviews (1-2 words): {(words.between(1, 2)).sum()}")
print(f"Checkbox-template reviews (contain ☐ or ☑): {text.str.contains('☐|☑').sum()}")
print(f"Mostly non-Latin text (under 50% Latin letters): "
      f"{(text.str.count(r'[A-Za-z]') / text.str.len().clip(lower=1) < 0.5).sum()}")

# Review length in words
print("\nWords per review:")
print(words.describe().round(1).to_string())

DATE RANGE:
Earliest: 2024-01-01 06:47:12, Latest: 2025-12-31 22:54:31

SCRAPE FILTER CHECKS:
voted_up
False    6612
Name: count, dtype: int64
language
english    6612
Name: count, dtype: int64

REVIEWS PER GAME:
group      name                               
disclosed  Night Club Simulator                    41
control    Pharmacy Simulator                      41
disclosed  Cinema Simulator 2025                   45
           Hotel Business Simulator                48
           Shawarma Legend                         56
           Esports Godfather                       56
           FACEMINER                               85
           Serum: Toxic Survival                   85
           Chernobylite 2: Exclusion Zone          87
control    Trash Goblin                           112
disclosed  Lost Rift                              119
control    Tavern Manager Simulator               124
           Ultimate Fishing Simulator 2           128
           Motel Manager Simulator    

## Observations
- 6,612 reviews from the 30 originally selected games, 29 columns. The two replacement games (Step 1) were scraped separately, so they appear under "Games with no reviews" here.
- The scrape filters worked! Every review is negative, tagged English, and dated Jan 1, 2024 – Dec 31, 2025.
- Disclosed games have about half as many reviews (2,250 vs. 4,362), consistent with the review gap found in prior research. Four control games hit the 500-review cap. Sampling up to 100 per game keeps large games from dominating.
- Counts can differ by one from the earlier summary run (e.g. Motel Manager Simulator: 131 vs. 132) because Steam data changes over time. All analysis uses this dated snapshot.
- 289 reviews overlap with the pilot (dev set): all of DREADZONE (100 of 187), TRIBES 3 (100 of 464), Hotel Business Simulator (48 of 48), and Night Club Simulator (41 of 41). Removing them leaves Hotel Business and Night Club with no test reviews.
    - Decision: both are replaced with two other low-rated AI-disclosed games (Step 1), so the 312 pilot reviews stay a clean dev set.
- 277 reviews (4.2%) are only 1–2 words, and 8 are checkbox templates. These are kept: they are real negative reviews, labeled other if they give no reason.
- 53 reviews (0.8%) are mostly non-Latin characters. These could be other languages tagged as English, or ASCII art; they are inspected before deciding whether to remove them.
- Reviews have a median of 48 words (mean 100, max 1,541). The long tail matters for the cost of the Claude step.

---

## Cleaning

In [11]:
df_clean = df.copy()
print("Starting rows:", len(df_clean))

Starting rows: 6612


## Step 1: Replace Games With No Test Reviews

In [12]:
# 1. Replace games whose reviews are all in the pilot (dev set)
n_before = len(df_clean)
print("Step 1 (replace games) - Starting rows:", n_before)

# Replacement scrape (already run on Nov 4, 2026):
# %run steam_reviews_scraper.py --appid-file data/game_selection/games_replacements.csv --max-per-game 500 --out data/reviews_raw_replacements.csv

dropped_ids = [2738870, 2943670]  # Hotel Business Simulator, Night Club Simulator
replacements = pd.read_csv("data/reviews_raw_replacements.csv")
expected_drop = df_clean["appid"].isin(dropped_ids).sum()

df_clean = df_clean[~df_clean["appid"].isin(dropped_ids)]
df_clean = pd.concat([df_clean, replacements], ignore_index=True)
n_after = len(df_clean)

# Assert the swap: old games gone, new reviews added, review set matches the 30 final games
print("Step 1 (replace games) - Ending rows:", n_after,
      f"(dropped {expected_drop}, added {len(replacements)})")
assert n_after == n_before - expected_drop + len(replacements)
assert not df_clean["appid"].isin(dropped_ids).any(), "Dropped games should be gone"
assert set(df_clean["appid"]) == set(games["appid"]), "Reviews should cover exactly the 30 final games"
assert set(replacements.columns) == set(df.columns), "Replacement scrape should have the same columns"
assert df_clean["recommendationid"].is_unique, "Review IDs should be unique"

# Why: All in-window reviews for Hotel Business Simulator and Night Club Simulator were in the pilot,
# which is the dev set and can't be used for testing. Replacing them with two low-rated AI-disclosed
# management sims (REAL ESTATE Simulator, The Last FARMER) keeps 5 games in each rating tier and the same genre mix.

Step 1 (replace games) - Starting rows: 6612
Step 1 (replace games) - Ending rows: 6623 (dropped 89, added 100)


## Step 2: Parse Review Dates

In [13]:
# 2. Parse review dates
n_before = len(df_clean)
print("Step 2 (dates) - Starting rows:", n_before)

df_clean["created"] = pd.to_datetime(df_clean["timestamp_created"], unit="s")

n_after = len(df_clean)
print("Step 2 (dates) - Ending rows:", n_after)
print(f"Date range: {df_clean['created'].min()} to {df_clean['created'].max()}")

# Assert row count unchanged, every date parsed, and all dates inside the study window
assert n_after == n_before, "Parsing dates shouldn't change row count"
assert df_clean["created"].notna().all(), "Some dates failed to parse"
assert df_clean["created"].between("2024-01-01", "2025-12-31 23:59:59").all(), "Reviews outside the study window"

# Why: created was read from the CSV as text, so it can't be sorted or grouped by month. Parsing it from the
# Unix timestamp gives a true datetime and confirms Steam's date filter also held for the replacement scrape.

Step 2 (dates) - Starting rows: 6623
Step 2 (dates) - Ending rows: 6623
Date range: 2024-01-01 06:47:12 to 2025-12-31 22:54:31


## Step 3: Validate Scrape Filters

In [14]:
# 3. Validate scrape filters
n_before = len(df_clean)
print("Step 3 (filters) - Starting rows:", n_before)
print(df_clean["voted_up"].value_counts(dropna=False))
print(df_clean["language"].value_counts(dropna=False))
n_after = len(df_clean)
print("Step 3 (filters) - Ending rows:", n_after)

# Assert every review is negative and English, including the replacement scrape
assert n_after == n_before
assert (df_clean["voted_up"] == False).all(), "Found a positive review"
assert (df_clean["language"] == "english").all(), "Found a non-English review"

# Why: The study only covers English negative reviews. If Steam's API ever returned other reviews, they would
# silently mix into the sample, so this confirms the filters held for both scrapes.

Step 3 (filters) - Starting rows: 6623
voted_up
False    6623
Name: count, dtype: int64
language
english    6623
Name: count, dtype: int64
Step 3 (filters) - Ending rows: 6623


## Step 4: Remove Pilot (Dev Set) Reviews

In [15]:
# 4. Remove pilot (dev set) reviews
n_before = len(df_clean)
print("Step 4 (pilot) - Starting rows:", n_before)

in_pilot = df_clean["recommendationid"].isin(pilot["recommendationid"])
expected_drop = in_pilot.sum()
print(df_clean[in_pilot].groupby("name").size().to_string())
df_clean = df_clean[~in_pilot].reset_index(drop=True)

n_after = len(df_clean)
print("Step 4 (pilot) - Ending rows:", n_after, f"(dropped {n_before - n_after})")

# Assert the expected rows were removed and no pilot review remains
assert n_after == n_before - expected_drop
assert not df_clean["recommendationid"].isin(pilot["recommendationid"]).any(), "Pilot reviews remain"

# Why: The pilot reviews were used to design the keyword search and will be used to tune the Claude prompt.
# Testing on them would overstate how well both methods work, so they are kept out of all test data.

Step 4 (pilot) - Starting rows: 6623
name
DREADZONE           100
TRIBES 3: Rivals    100
Step 4 (pilot) - Ending rows: 6423 (dropped 200)


## Inspect Mostly Non-Latin Reviews

In [16]:
# Look at reviews with under 50% Latin letters before deciding whether to remove them
text = df_clean["review"].str.strip()
latin_share = text.str.count(r"[A-Za-z]") / text.str.len().clip(lower=1)
non_latin = df_clean[latin_share < 0.5]

print(f"Mostly non-Latin reviews: {len(non_latin)}")
print(non_latin.groupby(["group", "name"]).size().to_string())
print()
for review in non_latin["review"].head(25):
    print("-", review[:120].replace("\n", " "))

Mostly non-Latin reviews: 51
group      name                               
control    Aloft                                   1
           Drug Dealer Simulator 2                10
           FBC: Firebreak                          4
           Farming Simulator 25                    3
           Kebab Chefs! - Restaurant Simulator     2
           Killing Floor 3                         5
           Sengoku Dynasty                         4
           Sniper Elite: Resistance                3
           Tactical Breach Wizards                 2
           Tavern Manager Simulator                2
           Trash Goblin                            1
           Ultimate Fishing Simulator 2            3
disclosed  DREADZONE                               2
           Esports Godfather                       2
           FACEMINER                               1
           Serum: Toxic Survival                   1
           Shawarma Legend                         3
           TRIBES 3: Ri